# Advanced Module 4 · Real-World Agents
Three complete agents doing real jobs — Travel Booking, Customer Support, Research Summarizer.
These become the building blocks for Module 5.

---
> **Setup:** Run the first two cells before anything else.

In [ ]:
%pip install -q google-genai groq python-dotenv

In [1]:
import os, json, time, re, getpass, random
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
api_key  = os.environ.get('GEMINI_API_KEY') or getpass.getpass('Paste your Gemini API key: ')
groq_key = (os.environ.get('Groq_key') or getpass.getpass('Paste your Groq API key: ')).strip()

client      = genai.Client(api_key=api_key)
groq_client = Groq(api_key=groq_key)
MODEL       = 'gemini-3.1-flash-lite'
GROQ_FAST   = 'llama-3.1-8b-instant'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s')
                time.sleep(wait)
            elif '500' in msg or 'INTERNAL' in msg:
                time.sleep(10 * (attempt + 1))
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen = client.models.generate_content
client.models.generate_content = lambda *a, **kw: _call_with_retry(_raw_gen, *a, **kw)

print('✅ Setup complete | Gemini:', MODEL, '| Groq fast:', GROQ_FAST)

✅ Setup complete | Gemini: gemini-3.1-flash-lite | Groq fast: llama-3.1-8b-instant


In [2]:
try:
    _r = client.models.generate_content(model=MODEL,
        contents='Reply with exactly the words: Hello LLM course', config=cfg(temperature=0.0))
    print('✅ API working:', get_text(_r))
except Exception as e:
    print('❌ Error:', e)

✅ API working: Hello LLM course


---
## Shared Agent Runner

All three agents below use the same ReAct loop from Module 2.  
Each agent only differs in its **system prompt**, **tool set**, and **tool implementations**.

In [3]:
def run_agent(
    task: str,
    system_prompt: str,
    tool_schema: types.Tool,
    tool_map: dict,
    max_steps: int = 10,
    label: str = 'Agent',
    verbose: bool = True
) -> str:
    """Generic ReAct agent runner — same loop from Module 2."""
    conversation = [
        types.Content(role='user', parts=[types.Part(
            text=f'[SYSTEM]: {system_prompt}\n\n[USER TASK]: {task}'
        )])
    ]
    total_tokens = 0

    if verbose:
        print('\n' + '═' * 65)
        print(f'🤖 {label}')
        print(f'📋 Task: {task[:100]}...' if len(task) > 100 else f'📋 Task: {task}')
        print('═' * 65)

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL, contents=conversation,
            config=cfg(tools=[tool_schema], temperature=0.2)
        )
        total_tokens += response.usage_metadata.total_token_count

        fcs = [
            p.function_call
            for p in response.candidates[0].content.parts
            if hasattr(p, 'function_call') and p.function_call
        ]

        if not fcs:
            final = get_text(response)
            if verbose:
                print(f'\n[Step {step}] ✅ FINAL ANSWER:')
                print(final)
                print(f'\n📊 Steps: {step} | Tokens: {total_tokens}')
            return final

        results = []
        for fc in fcs:
            if fc.name not in tool_map:
                results.append({'error': f'Unknown tool: {fc.name}'})
                continue
            try:
                result = tool_map[fc.name](**dict(fc.args))
                results.append(result)
                if verbose:
                    print(f'[Step {step}] 🔧 {fc.name}({dict(fc.args)})')
                    print(f'           → {json.dumps(result)[:120]}')
            except Exception as e:
                results.append({'error': str(e)})
                if verbose:
                    print(f'[Step {step}] ❌ {fc.name} ERROR: {e}')

        conversation.append(response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(name=fc.name, response={'result': r}))
            for fc, r in zip(fcs, results)
        ]))

    return 'Max steps reached.'

print('✅ Agent runner ready')

✅ Agent runner ready


---
## Agent 1 — Travel Booking Agent

Handles end-to-end trip planning: flight search, hotel search, constraint reasoning, booking.

In [4]:
# Tool implementations
def search_flights(origin: str, destination: str, date: str) -> dict:
    options = [
        {'flight_id': 'SK201', 'airline': 'SkyWings',  'price_usd': 580, 'duration_h': 9,  'class': 'Economy'},
        {'flight_id': 'AE445', 'airline': 'AeroEast',  'price_usd': 640, 'duration_h': 11, 'class': 'Economy'},
        {'flight_id': 'SW900', 'airline': 'SwiftAir',  'price_usd': 520, 'duration_h': 10, 'class': 'Economy'},
        {'flight_id': 'LX101', 'airline': 'LuxAir',    'price_usd': 1200,'duration_h': 8,  'class': 'Business'},
    ]
    return {'route': f'{origin} → {destination}', 'date': date, 'options': options}

def search_hotels(city: str, checkin: str, checkout: str) -> dict:
    options = [
        {'hotel_id': 'H01', 'name': 'CityInn',        'price_per_night_usd': 85,  'rating': 3.8, 'stars': 3},
        {'hotel_id': 'H02', 'name': 'ParkView Grand', 'price_per_night_usd': 140, 'rating': 4.5, 'stars': 4},
        {'hotel_id': 'H03', 'name': 'BudgetStay',     'price_per_night_usd': 55,  'rating': 3.2, 'stars': 2},
        {'hotel_id': 'H04', 'name': 'Royal Suite',    'price_per_night_usd': 350, 'rating': 4.9, 'stars': 5},
    ]
    return {'city': city, 'checkin': checkin, 'checkout': checkout, 'options': options}

def book_itinerary(flight_id: str, hotel_id: str, passenger_name: str) -> dict:
    import random, string
    booking_ref = ''.join(random.choices(string.ascii_uppercase + string.digits, k=8))
    flight_prices = {'SK201': 580, 'AE445': 640, 'SW900': 520, 'LX101': 1200}
    hotel_prices  = {'H01': 85, 'H02': 140, 'H03': 55, 'H04': 350}
    return {
        'booking_ref':  booking_ref,
        'status':       'CONFIRMED',
        'passenger':    passenger_name,
        'flight_id':    flight_id,
        'hotel_id':     hotel_id,
        'flight_cost':  flight_prices.get(flight_id, 0),
        'hotel_cost':   hotel_prices.get(hotel_id, 0),
        'message':      f'Booking confirmed! Reference: {booking_ref}'
    }

def get_exchange_rate(from_currency: str, to_currency: str) -> dict:
    rates = {'USD': 1.0, 'EUR': 0.93, 'GBP': 0.79, 'INR': 83.5, 'AED': 3.67}
    rate  = rates.get(to_currency.upper(), 1.0) / rates.get(from_currency.upper(), 1.0)
    return {'from': from_currency, 'to': to_currency, 'rate': round(rate, 4)}

BOOKING_TOOLS = {
    'search_flights':   search_flights,
    'search_hotels':    search_hotels,
    'book_itinerary':   book_itinerary,
    'get_exchange_rate': get_exchange_rate,
}

booking_schema = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='search_flights',
        description='Search for available flights. Returns multiple options with prices and duration.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['origin','destination','date'],
            properties={
                'origin':      types.Schema(type=types.Type.STRING, description='Departure city'),
                'destination': types.Schema(type=types.Type.STRING, description='Arrival city'),
                'date':        types.Schema(type=types.Type.STRING, description='Travel date YYYY-MM-DD'),
            })
    ),
    types.FunctionDeclaration(
        name='search_hotels',
        description='Search for hotels in a city for given dates. Returns options with price and rating.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['city','checkin','checkout'],
            properties={
                'city':     types.Schema(type=types.Type.STRING),
                'checkin':  types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                'checkout': types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
            })
    ),
    types.FunctionDeclaration(
        name='book_itinerary',
        description='Confirm a booking with a specific flight and hotel. Returns a booking reference.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['flight_id','hotel_id','passenger_name'],
            properties={
                'flight_id':      types.Schema(type=types.Type.STRING, description='Flight ID from search_flights'),
                'hotel_id':       types.Schema(type=types.Type.STRING, description='Hotel ID from search_hotels'),
                'passenger_name': types.Schema(type=types.Type.STRING),
            })
    ),
    types.FunctionDeclaration(
        name='get_exchange_rate',
        description='Get current exchange rate between two currencies.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['from_currency','to_currency'],
            properties={
                'from_currency': types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
                'to_currency':   types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
            })
    ),
])

BOOKING_SYSTEM_PROMPT = """You are a smart travel booking assistant.
Help the user find and book the best trip based on their constraints (budget, dates, preferences).
Always search for flights and hotels before booking. Reason about the options and pick the best match.
When booking, confirm all details with the user's name."""

print('✅ Booking Agent tools defined')

✅ Booking Agent tools defined


In [5]:
run_agent(
    task=(
        "I'm Priya Sharma and I want to fly from Mumbai to Berlin on 2025-08-15 for 3 nights. "
        "My budget is $2000 total. I prefer a 4-star hotel. Book the best option for me."
    ),
    system_prompt=BOOKING_SYSTEM_PROMPT,
    tool_schema=booking_schema,
    tool_map=BOOKING_TOOLS,
    label='Travel Booking Agent'
)


═════════════════════════════════════════════════════════════════
🤖 Travel Booking Agent
📋 Task: I'm Priya Sharma and I want to fly from Mumbai to Berlin on 2025-08-15 for 3 nights. My budget is $2...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 search_flights({'destination': 'Berlin', 'date': '2025-08-15', 'origin': 'Mumbai'})
           → {"route": "Mumbai \u2192 Berlin", "date": "2025-08-15", "options": [{"flight_id": "SK201", "airline": "SkyWings", "price
[Step 2] 🔧 search_hotels({'checkin': '2025-08-15', 'checkout': '2025-08-18', 'city': 'Berlin'})
           → {"city": "Berlin", "checkin": "2025-08-15", "checkout": "2025-08-18", "options": [{"hotel_id": "H01", "name": "CityInn",
[Step 3] 🔧 book_itinerary({'passenger_name': 'Priya Sharma', 'hotel_id': 'H02', 'flight_id': 'SK201'})
           → {"booking_ref": "KD45WXSF", "status": "CONFIRMED", "passenger": "Priya Sharma", "flight_id": "SK201", "hotel_id": "H02",

[Step 4] ✅ FINAL ANSWER:
Hello Priy

'Hello Priya Sharma, I have successfully booked your trip to Berlin!\n\nHere are the details of your booking:\n*   **Flight:** SkyWings (Flight ID: SK201) from Mumbai to Berlin on 2025-08-15.\n*   **Hotel:** ParkView Grand (4-star, Hotel ID: H02) for 3 nights (2025-08-15 to 2025-08-18).\n*   **Total Cost:** $1,000 ($580 for the flight + $420 for 3 nights at the hotel), which is well within your $2,000 budget.\n*   **Booking Reference:** KD45WXSF\n\nEnjoy your trip to Berlin!'

---
## Agent 2 — Customer Support Agent

Handles complaints, refunds, and escalations — with automatic escalation to a human when needed.

In [6]:
def lookup_order(order_id: str) -> dict:
    orders = {
        'ORD-1001': {'status': 'delivered', 'item': 'Laptop Pro 15"',  'price': 1299, 'delivery_date': '2025-07-10'},
        'ORD-1002': {'status': 'in_transit', 'item': 'Wireless Headphones', 'price': 249, 'eta': '2025-07-20'},
        'ORD-1003': {'status': 'cancelled',  'item': 'Smart Watch',     'price': 399, 'cancel_reason': 'out_of_stock'},
    }
    return orders.get(order_id, {'error': f'Order {order_id} not found'})

def check_refund_policy(reason: str) -> dict:
    policies = {
        'defective':  {'eligible': True,  'window_days': 30, 'method': 'full refund or replacement'},
        'not_needed': {'eligible': True,  'window_days': 14, 'method': 'refund minus 10% restocking fee'},
        'late':       {'eligible': True,  'window_days': 7,  'method': 'full refund'},
        'fraud':      {'eligible': True,  'window_days': 60, 'method': 'immediate full refund'},
        'other':      {'eligible': False, 'window_days': 0,  'method': 'escalate to human agent'},
    }
    return policies.get(reason.lower(), policies['other'])

def raise_ticket(order_id: str, issue_type: str, description: str) -> dict:
    import random
    ticket_id = f'TKT-{random.randint(10000, 99999)}'
    return {'ticket_id': ticket_id, 'status': 'open', 'priority': 'normal', 'sla_hours': 24}

def escalate_to_human(reason: str, order_id: str) -> dict:
    return {
        'escalated':    True,
        'queue':        'tier-2-support',
        'wait_time_min': 15,
        'message':      f'Your case ({order_id}) has been escalated. A human agent will contact you shortly. Reason: {reason}'
    }

SUPPORT_TOOLS = {
    'lookup_order':     lookup_order,
    'check_refund_policy': check_refund_policy,
    'raise_ticket':     raise_ticket,
    'escalate_to_human': escalate_to_human,
}

support_schema = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='lookup_order', description='Look up order details by order ID.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['order_id'],
            properties={'order_id': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='check_refund_policy',
        description='Check refund policy for a given return reason.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['reason'],
            properties={'reason': types.Schema(type=types.Type.STRING, description='defective / not_needed / late / fraud / other')})
    ),
    types.FunctionDeclaration(
        name='raise_ticket',
        description='Create a support ticket for an order issue.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['order_id','issue_type','description'],
            properties={
                'order_id':    types.Schema(type=types.Type.STRING),
                'issue_type':  types.Schema(type=types.Type.STRING),
                'description': types.Schema(type=types.Type.STRING),
            })
    ),
    types.FunctionDeclaration(
        name='escalate_to_human',
        description='Escalate to a human agent when the issue is too complex or sensitive.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['reason','order_id'],
            properties={
                'reason':   types.Schema(type=types.Type.STRING, description='Why escalation is needed'),
                'order_id': types.Schema(type=types.Type.STRING),
            })
    ),
])

SUPPORT_SYSTEM_PROMPT = """You are a professional customer support agent.
Be empathetic and solution-focused. Always look up the order first before making decisions.
If the issue is complex, uncertain, or the customer is very upset, escalate to a human agent.
Always raise a ticket for any issue so it's tracked."""

print('✅ Customer Support Agent tools defined')

✅ Customer Support Agent tools defined


In [7]:
run_agent(
    task=(
        "I received my laptop from order ORD-1001 and it's completely broken — "
        "the screen doesn't turn on. I paid $1299 for this and I'm furious. "
        "I want a full refund immediately!"
    ),
    system_prompt=SUPPORT_SYSTEM_PROMPT,
    tool_schema=support_schema,
    tool_map=SUPPORT_TOOLS,
    label='Customer Support Agent'
)


═════════════════════════════════════════════════════════════════
🤖 Customer Support Agent
📋 Task: I received my laptop from order ORD-1001 and it's completely broken — the screen doesn't turn on. I ...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 lookup_order({'order_id': 'ORD-1001'})
           → {"status": "delivered", "item": "Laptop Pro 15\"", "price": 1299, "delivery_date": "2025-07-10"}
[Step 2] 🔧 check_refund_policy({'reason': 'defective'})
           → {"eligible": true, "window_days": 30, "method": "full refund or replacement"}
[Step 3] 🔧 raise_ticket({'issue_type': 'defective_product', 'order_id': 'ORD-1001', 'description': "Customer received a broken laptop (screen won't turn on). Customer is very upset and requesting a full refund."})
           → {"ticket_id": "TKT-75315", "status": "open", "priority": "normal", "sla_hours": 24}
[Step 4] 🔧 escalate_to_human({'reason': 'Customer is extremely upset and demanding an immediate refund for a hig

'I am incredibly sorry to hear that your new laptop arrived in this condition. I completely understand your frustration—spending $1299 on a device only to have it arrive broken is unacceptable, and I sincerely apologize for this experience.\n\nI have taken the following steps to resolve this for you:\n\n1.  **Ticket Created:** I have opened a support ticket for your issue (Ticket ID: **TKT-75315**) so that we have a formal record of this incident.\n2.  **Escalation:** Because this is a high-value item and I understand how urgent this is for you, I have escalated your case directly to our senior support team. They are best equipped to process your refund request immediately.\n\nA human agent from our tier-2 support team will contact you within the next 15 minutes to finalize the refund process. Please keep an eye on your email or phone for their update. Thank you for your patience while we make this right.'

---
## Agent 3 — Research Summarizer Agent

Fetches articles, extracts key claims, cross-checks for consistency, and writes a structured report.

In [8]:
# Stub article database
ARTICLE_DB = {
    'https://techblog.example.com/ai-agents-2025': {
        'title': 'The Rise of AI Agents in 2025',
        'text': (
            'AI agents are now handling 40% of customer support queries at Fortune 500 companies. '
            'Multi-agent systems have shown 3x productivity gains in software development tasks. '
            'However, agent hallucination rates remain at 8-12% without proper grounding.'
        )
    },
    'https://research.example.com/llm-productivity': {
        'title': 'LLM Impact on Developer Productivity',
        'text': (
            'Studies show developers using AI coding assistants complete tasks 55% faster. '
            'Code review time has dropped by 30% with LLM-assisted analysis. '
            'AI agents account for 25% of new startup codebases according to Y Combinator.'
        )
    },
    'https://safety.example.com/agent-risks': {
        'title': 'Safety Risks in Agentic AI Systems',
        'text': (
            'Prompt injection attacks have been demonstrated on 15 production agent systems. '
            'Agents with unrestricted tool access caused $2.3M in accidental costs in 2024. '
            'Researchers recommend strict tool allowlists and human-in-the-loop for irreversible actions.'
        )
    }
}

def fetch_article(url: str) -> dict:
    article = ARTICLE_DB.get(url)
    if not article:
        return {'error': f'Article not found: {url}'}
    return {'url': url, 'title': article['title'], 'content': article['text'], 'word_count': len(article['text'].split())}

def extract_key_claims(text: str) -> dict:
    """Use Groq (fast) to extract key claims from text."""
    resp = groq_client.chat.completions.create(
        model=GROQ_FAST,
        messages=[{
            'role': 'user',
            'content': f'Extract 2-3 key factual claims from this text as a JSON array of strings:\n\n{text}'
        }],
        max_tokens=256,
        temperature=0.0
    )
    content = resp.choices[0].message.content
    # Try to parse JSON, fall back to lines
    try:
        match = re.search(r'\[.*?\]', content, re.DOTALL)
        claims = json.loads(match.group(0)) if match else [content]
    except Exception:
        claims = [line.strip() for line in content.splitlines() if line.strip()]
    return {'claims': claims, 'count': len(claims)}

def fact_check(claim: str) -> dict:
    """Use Groq to assess claim credibility."""
    resp = groq_client.chat.completions.create(
        model=GROQ_FAST,
        messages=[{
            'role': 'user',
            'content': (
                f'Rate this claim as LIKELY_TRUE, UNCERTAIN, or LIKELY_FALSE, with a 1-sentence reason.\n\n'
                f'Claim: {claim}\n\nFormat: {{"verdict": "...", "reason": "..."}}'
            )
        }],
        max_tokens=128,
        temperature=0.0
    )
    content = resp.choices[0].message.content
    try:
        match = re.search(r'\{.*?\}', content, re.DOTALL)
        result = json.loads(match.group(0)) if match else {'verdict': 'UNCERTAIN', 'reason': content}
    except Exception:
        result = {'verdict': 'UNCERTAIN', 'reason': content[:100]}
    return {**result, 'claim': claim}

def list_available_articles(topic: str) -> dict:
    """Return available article URLs for a topic."""
    return {
        'topic': topic,
        'available_urls': list(ARTICLE_DB.keys()),
        'count': len(ARTICLE_DB)
    }

RESEARCH_TOOLS = {
    'list_available_articles': list_available_articles,
    'fetch_article':           fetch_article,
    'extract_key_claims':      extract_key_claims,
    'fact_check':              fact_check,
}

research_schema = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='list_available_articles',
        description='List available article URLs for a given research topic.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['topic'],
            properties={'topic': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='fetch_article',
        description='Fetch the full text of an article given its URL.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['url'],
            properties={'url': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='extract_key_claims',
        description='Extract 2-3 key factual claims from a block of text.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['text'],
            properties={'text': types.Schema(type=types.Type.STRING)})
    ),
    types.FunctionDeclaration(
        name='fact_check',
        description='Assess the credibility of a single factual claim.',
        parameters=types.Schema(type=types.Type.OBJECT, required=['claim'],
            properties={'claim': types.Schema(type=types.Type.STRING)})
    ),
])

RESEARCH_SYSTEM_PROMPT = """You are a rigorous research analyst.
For any research task: list available articles, fetch them, extract key claims, and fact-check each one.
Synthesize a structured report that includes verified claims, uncertain claims, and a summary."""

print('✅ Research Summarizer Agent tools defined')

✅ Research Summarizer Agent tools defined


In [9]:
run_agent(
    task='Research the current state of AI agents in 2025. Fetch available articles, extract claims, fact-check them, and write a structured summary.',
    system_prompt=RESEARCH_SYSTEM_PROMPT,
    tool_schema=research_schema,
    tool_map=RESEARCH_TOOLS,
    label='Research Summarizer Agent',
    max_steps=12
)


═════════════════════════════════════════════════════════════════
🤖 Research Summarizer Agent
📋 Task: Research the current state of AI agents in 2025. Fetch available articles, extract claims, fact-chec...
═════════════════════════════════════════════════════════════════
[Step 1] 🔧 list_available_articles({'topic': 'current state of AI agents 2025'})
           → {"topic": "current state of AI agents 2025", "available_urls": ["https://techblog.example.com/ai-agents-2025", "https://
[Step 2] 🔧 fetch_article({'url': 'https://techblog.example.com/ai-agents-2025'})
           → {"url": "https://techblog.example.com/ai-agents-2025", "title": "The Rise of AI Agents in 2025", "content": "AI agents a
[Step 3] 🔧 fetch_article({'url': 'https://research.example.com/llm-productivity'})
           → {"url": "https://research.example.com/llm-productivity", "title": "LLM Impact on Developer Productivity", "content": "St
[Step 4] 🔧 fetch_article({'url': 'https://safety.example.com/agent-risks'})
    

'# Research Report: The State of AI Agents in 2025\n\nThis report synthesizes findings regarding the current landscape of AI agents, focusing on productivity impacts, adoption, and security risks.\n\n## Verified and Supported Claims\n*   **Developer Productivity Gains:** Research consistently indicates that AI coding assistants significantly improve developer efficiency. Studies suggest that developers using these tools can complete tasks up to 55% faster, with code review times decreasing by approximately 30%.\n\n## Uncertain Claims\n*   **Customer Support Adoption:** The claim that AI agents handle 40% of customer support queries at Fortune 500 companies remains unverified. While adoption is high, industry-wide data is fragmented, and specific percentages vary significantly by sector.\n*   **Financial Impact of Agent Errors:** The report of $2.3M in accidental costs caused by agents with unrestricted tool access in 2024 lacks sufficient public documentation or verifiable industry-wid

In [ ]:
# ✏️  Add a new tool to one of the agents above.
#     Ideas for Booking Agent: check_baggage_policy(airline), get_airport_info(city)
#     Ideas for Support Agent: get_loyalty_points(customer_id), send_voucher(email, amount)
#     Ideas for Research Agent: summarize_with_bullets(text), compare_sources(url1, url2)

def my_new_tool(input_param: str) -> dict:
    # TODO: implement your tool
    return {'result': f'Processed: {input_param}'}

print('Define your tool and add it to one of the TOOL_MAP dicts above, then re-run the agent.')

---
## Key Takeaways

| Agent | What makes it real-world? |
|---|---|
| Travel Booking | Constraint reasoning (budget, rating), multi-hop (search → select → book) |
| Customer Support | Empathy + escalation trigger, policy lookup, ticket tracking |
| Research Summarizer | Uses Groq for fast sub-tasks, fact-checks every claim before reporting |

| Pattern | Applies to |
|---|---|
| Same `run_agent()` loop for all 3 | Architecture is reusable; agents differ only in system prompt + tools |
| Groq for fast sub-tasks | Use fast small models for classification/extraction; Gemini for reasoning |
| These are worker agents | Module 5 will orchestrate them together under a single orchestrator |